# 02 – Huấn luyện M4 (Logistic Regression học trọng số)

Ghép hai file nhãn độc lập, tính kappa, chốt nhãn theo quy tắc (lệch >= 2 bậc: thảo luận; lệch 1 bậc: trung bình làm tròn xuống), nhãn nhị phân "phù hợp" khi nhãn >= 2, chia **GroupKFold 5 theo `candidate_id`** để không rò rỉ.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

# Có thể đổi bằng biến môi trường khi chạy thử trên dữ liệu khác
DATA = Path(os.getenv("TH3_DATA_DIR", ROOT / "backend/seed/data"))
LABELS = Path(os.getenv("TH3_LABELS_DIR", ROOT / "backend/seed/labels"))
OUTDIR = Path(os.getenv("TH3_OUT_DIR", ROOT / "docs/report/tables"))
MODELS_OUT = Path(os.getenv("TH3_MODELS_OUT", ROOT / "backend/models_store"))
OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from app.ai import datasets, metrics
from app.ai.embedder import get_embedder

emb = get_embedder()
print("Embedder:", emb.name)  # st:... là model thật; hash-384 chỉ để chạy thử, KHÔNG dùng số liệu để báo cáo
jobs = datasets.load_jobs(DATA / "jobs.csv")
cands = datasets.load_candidates(DATA / "candidates_generated.csv")
datasets.attach_embeddings(jobs, emb)
datasets.attach_embeddings(cands, emb)

LABEL_A = os.getenv("TH3_LABELS_A", "labels_pairs_hung.csv")
LABEL_B = os.getenv("TH3_LABELS_B", "labels_pairs_hoanganh.csv")
raw = datasets.load_labels(LABELS / LABEL_A, LABELS / LABEL_B)
col_a, col_b = [c for c in raw.columns if c.startswith("label_")]
print(f"{len(raw)} cặp có đủ nhãn của cả hai người")


In [ ]:
kappa = metrics.weighted_kappa(raw[col_a].to_numpy(), raw[col_b].to_numpy())
print(f"Cohen's kappa (trọng số bình phương) = {kappa:.3f}  (mục tiêu >= 0,6)")

raw["label"] = metrics.merge_labels(raw[col_a].to_numpy(), raw[col_b].to_numpy())
disputed = raw[raw["label"] < 0]
print(f"{len(disputed)} cặp lệch >= 2 bậc cần hai người thảo luận và chốt lại")
disputed.to_csv(OUTDIR / "labels_disputed.csv", index=False)
labels = raw[raw["label"] >= 0].reset_index(drop=True)  # SAU KHI thảo luận: cập nhật file nhãn rồi chạy lại ô này
labels[["candidate_id", "job_id", "label"]].to_csv(LABELS / "labels_pairs_final.csv", index=False)
pd.DataFrame([{"kappa_quadratic": kappa, "pairs": len(raw), "disputed": len(disputed)}]).to_csv(OUTDIR / "kappa.csv", index=False)


In [ ]:
from app.ai import evaluate, train
from sklearn.metrics import roc_auc_score

X, feats = evaluate.build_feature_matrix(labels, cands, jobs, emb)
y = (labels["label"] >= 2).astype(int).to_numpy()
groups = labels["candidate_id"].to_numpy()
print("Số cặp:", len(y), "| tỉ lệ phù hợp:", round(y.mean(), 3), "| số ứng viên:", len(set(groups)))

oof = train.cross_val_scores(X, y, groups, n_splits=5)
print("ROC-AUC trên fold kiểm tra (GroupKFold):", round(roc_auc_score(y, oof), 4))
pd.DataFrame({"candidate_id": labels["candidate_id"], "job_id": labels["job_id"], "m4_oof_score": oof * 100}).to_csv(OUTDIR / "m4_oof.csv", index=False)

In [ ]:
model = train.fit_m4(X, y)
coefs = pd.DataFrame({"feature": list(train.FEATURE_NAMES), "m4_coef": model.coef_[0],
                      "m3_hand_weight": [train.M3_DEFAULT_WEIGHTS[n] for n in train.FEATURE_NAMES]})
coefs.to_csv(OUTDIR / "m4_coefficients.csv", index=False)
coefs.round(4)

In [ ]:
version = train.save_m4(model, MODELS_OUT / "m4.joblib", extra={"kappa": kappa, "n_pairs": int(len(y)), "embedder": emb.name})
print("Đã lưu", MODELS_OUT / "m4.joblib", "| phiên bản", version)

# Chỉ đặt UPDATE_M3 = True nếu bảng ở notebook 03 cho thấy M4 tốt hơn M3 (hệ số dương chuẩn hoá về tổng 1).
UPDATE_M3 = False
if UPDATE_M3:
    w = train.m3_weights_from_m4(model)
    train.save_m3_weights(w, MODELS_OUT / "m3_weights.json", f"m3-from-{version}")
    print("Đã cập nhật trọng số M3:", w)